# 01 — Ingest & retrieve (Phase 1)

**Goal of this notebook:** take a handful of source PDFs, chunk them, embed the
chunks locally, store them in Chroma, and show retrieval working end to end —
ask a question, get back the most relevant passages *with citations* (document
title + page number). No LLM and no API yet; Phase 1 is about proving the
retrieval half is sound before anything is wrapped in a service.

**How the code is organised.** The notebook orchestrates; the logic lives in
three small modules under `src/rag_tutoring/`, installed as an editable package:

| module | responsibility |
|---|---|
| `config.py` | paths + the few tunable knobs (embedding model, chunk size) |
| `ingest.py` | PDF → page-tagged, overlapping text chunks |
| `store.py`  | `VectorStore`: embed + store + query, behind a narrow interface |

That last seam is deliberate (see `docs/decisions/0001`): nothing here calls
Chroma directly, so swapping the store later touches one file.

> ⚠️ **Before committing this notebook, clear its outputs.** The retrieval cells
> print verbatim passages from copyrighted source PDFs — the same text `data/`
> is git-ignored to keep out of the repo. The closing cell has the one-liner.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from rag_tutoring import config
from rag_tutoring.ingest import chunk_pdf
from rag_tutoring.store import VectorStore

print("project root:  ", config.ROOT)
print("embedding model:", config.EMBEDDING_MODEL)
print("chunk size:    ", config.CHUNK_WORDS, "words,", config.CHUNK_OVERLAP_WORDS, "overlap")

## 1. The starter slice

The full corpus is 37 documents. Iterating on chunking against all of them is
slow, so this notebook works a five-paper slice chosen to tell one coherent
story — the road to modern NLP:

1. **word2vec** — *Efficient Estimation of Word Representations in Vector Space*
2. **GloVe** — *Global Vectors for Word Representation*
3. **Transformer** — *Attention Is All You Need*
4. **BERT** — *Pre-training of Deep Bidirectional Transformers*
5. **RAG** — *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*

The RAG paper is in here on purpose: it lets the tutor explain *its own*
architecture, which is a nice thing to demo. Scaling to the full corpus is just
a longer file list plus per-source-type chunk settings for the textbooks.

In [ ]:
PAPERS = config.DATA_RAW / "papers"
STARTER = [
    PAPERS / "Efficient Estimation of Word Representations in Vector Space.pdf",
    PAPERS / "GloVe- Global Vectors for Word Representation.pdf",
    PAPERS / "Attention Is All You Need.pdf",
    PAPERS / "BERT- Pre-training of Deep Bidirectional Transformers for Language Understanding.pdf",
    PAPERS / "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.pdf",
]

missing = [p.name for p in STARTER if not p.exists()]
assert not missing, f"missing starter PDFs: {missing}"
print(f"all {len(STARTER)} starter PDFs present")

## 2. Load & inspect

Confirm the text actually extracts before trusting any of it downstream. For
each paper we print the page count and a snippet, so a scanned or image-only
PDF (which would extract as near-empty) is obvious immediately.

In [ ]:
from rag_tutoring.ingest import load_pdf

for path in STARTER:
    pages = load_pdf(path)
    total_chars = sum(len(t) for _, t in pages)
    print(f"{path.stem[:55]:55s}  {len(pages):3d} pages  {total_chars:>7,} chars")

# Spot-check one paper's first page of extracted text.
sample_pages = load_pdf(STARTER[2])  # Attention Is All You Need
print("\n--- sample: first page of 'Attention Is All You Need' ---\n")
print(sample_pages[0][1][:600])

## 3. Chunk

Split each page into ~160-word windows with 40-word overlap. Chunking stays
inside a page so every chunk cites exactly one page. All five are papers, so
`source_type="paper"`; textbooks will get their own settings later.

In [ ]:
chunks = []
for path in STARTER:
    doc_chunks = chunk_pdf(path, source_type="paper")
    chunks.extend(doc_chunks)
    print(f"{path.stem[:55]:55s}  {len(doc_chunks):4d} chunks")

print(f"\ntotal chunks: {len(chunks)}")

# What one chunk actually looks like: text + the metadata that makes it citable.
c = chunks[len(chunks) // 2]
print(f"\n--- example chunk ---\nid:    {c.id}\nsource:{c.source} (p.{c.page}, {c.source_type})\ntext:  {c.text[:300]}...")

## 4. Embed & index

`VectorStore` loads the local sentence-transformers model, embeds each chunk,
and upserts it into a persistent Chroma collection (under `chroma/`,
git-ignored). The collection is named after the embedding model — a different
model means a different, incomparable index, so it gets its own collection.

*First run downloads the model (~90 MB) and embeds the slice — expect a minute
or two. Re-runs are fast; upsert overwrites by id rather than duplicating.*

In [ ]:
store = VectorStore()
print("collection:", store.collection_name)

# reset() first so re-running after a chunk-size change rebuilds cleanly
# instead of leaving stale chunks from the previous run behind.
store.reset()
store.add(chunks)
print("indexed chunks:", store.count())

## 5. Retrieve

The payoff. Each question is embedded with the same model and matched against
the index. Results come back as `Retrieved` objects carrying the citation
(`source`, `page`) and a **cosine similarity** in `[0, 1]` — higher is closer.

In [ ]:
def show(question, k=4):
    print(f"Q: {question}\n" + "-" * 78)
    for i, hit in enumerate(store.query(question, k=k), start=1):
        snippet = " ".join(hit.text.split())[:220]
        print(f"{i}. [{hit.score:.3f}] {hit.source} (p.{hit.page})\n   {snippet}...\n")

show("What is retrieval-augmented generation and how does it use a retriever?")

In [ ]:
show("How does multi-head self-attention work in the Transformer?")

In [ ]:
show("What problem do learned word embeddings like word2vec solve?")

## 6. Sanity check: is the score really cosine similarity?

Chroma's distance metric is set via collection metadata, and a silent fallback
to L2 would leave the *ranking* correct (embeddings are normalised) while making
the printed score meaningless. So verify it directly: query with the exact text
of a known chunk. The top hit must be that chunk, at similarity ≈ 1.0.

In [ ]:
probe = chunks[len(chunks) // 2]
top = store.query(probe.text, k=1)[0]

print(f"probe chunk: {probe.source} (p.{probe.page})")
print(f"top hit:     {top.source} (p.{top.page})  score={top.score:.4f}")

assert top.text == probe.text, "identity query did not return the probe chunk"
assert top.score > 0.99, f"cosine similarity should be ~1.0, got {top.score:.4f}"
print("\nOK — metric is cosine similarity and ordering is correct.")

## Where this goes next

**Works today:** end-to-end retrieval with citations over the starter slice,
behind a store interface that isn't tied to Chroma.

**Known limitations (expected, not bugs):**
- These are two-column PDFs; pypdf sometimes interleaves columns, so a snippet
  can read slightly jumbled. It's extraction reading-order, not the pipeline.
- Per-page chunking splits any passage that straddles a page break.
- Textbook chunks will reference figures the retriever can't see ("the green
  shape") — fine as long as the surrounding prose carries the meaning.

**Next steps (Phase 1):**
1. Scale to the full corpus; give textbooks their own chunk settings (larger
   windows, they're multi-topic and long).
2. Build the 20–30 question eval set with known-correct sources (Phase 2 gate).
3. Tune chunk size / overlap against that eval set — the loop is free because
   embeddings are local.

---
**Before committing, clear outputs** (the cells above print copyrighted source text):

```
jupyter nbconvert --to notebook --clear-output --inplace notebooks/01_ingest_and_retrieve.ipynb
```